# 解码阶段到底受什么限制 —— 检验一个假说

## 要检验的假说

2026-09-05 的量化对照里，GPTQ-Int4 比 fp16 快约 1.9 倍，**推翻了我"T4 无 Marlin 所以会更慢"的预测**。
我当时给的解释是：

> 1.5B 模型在 T4 上解码阶段大概率是**权重读取带宽受限**，INT4 权重约为 fp16 的 1/4，
> 省下的访存量压过了 kernel 效率的损失。

**当时我明确标注了这是假说、没有测量证据。** 这一轮就是来给它证据的。

## 怎么检验（不用 nsight，全部可直接测量）

若解码真的受权重带宽限制，那么 batch=1 时每生成一个 token 必须把**全部权重**读一遍，于是：

```
TPOT 理论下界 = 权重字节数 / 可达显存带宽
```

三件事都能测：

| 量 | 怎么拿 |
|---|---|
| 可达显存带宽 | 本 notebook 第 2 节：大张量 copy / 求和微基准，实测 GB/s |
| 权重字节数 | vLLM 启动日志的 `model weights take X GiB`（实际驻留，比文件大小准） |
| 实测 TPOT | 压测脚本在并发 1 下报的 TPOT p50 |

**判据：**

- 若三个变体的 `实测TPOT ÷ 理论下界` 都落在同一个量级（比如都在 1.5–2.5 倍之间），
  说明它们都贴着同一条带宽屋顶跑 → **假说成立**。
- 若 fp16 贴着屋顶、而 INT4 明显偏离（比值大得多），
  说明反量化开销吃掉了访存收益 → **假说至少不完整**。
- 若三者都远离屋顶（比如 5 倍以上），说明瓶颈根本不在权重带宽 → **假说被推翻**。

## 第二个可测量的推论

若低并发受带宽限制、高并发转为受算力限制，那么**量化的优势应当随并发升高而收窄**。
上一轮观察到的 1.88×（并发1）→ 1.47×（并发32）与这个方向一致。
本轮把 TPOT 随并发的曲线也画出来，看拐点在哪。

**不管结论是支持、部分支持还是推翻，照实写。**


## 0. 环境

In [ ]:
import subprocess
out = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
                      "--format=csv"], capture_output=True, text=True).stdout
print(out)
name = out.strip().splitlines()[-1].split(",")[0].strip()
print("显卡:", name)
print()
print("要与 2026-09-05 量化那轮可比，这里必须也是 Tesla T4。不是就别往下跑。")
print("T4 官方标称显存带宽 320 GB/s —— 但标称值不能直接用作屋顶，第 2 节实测可达值。")


## 1. 装 vLLM

沿用已验证的口径：**不装 torchaudio**（与 torch 版本错配会炸 transformers 导入链），
装完再卸一次以防依赖把它拉回来。


In [ ]:
import importlib.metadata as md_, subprocess, sys

CHECK = ["vllm", "aiohttp", "torchvision"]

def ver(p):
    try:
        return md_.version(p)
    except Exception:
        return None

def sh(c):
    return subprocess.run(c, capture_output=True, text=True)

missing = [p for p in CHECK if ver(p) is None]
print("缺失:", missing or "无")
if missing:
    print("装 vllm + aiohttp + torchvision（约 5-10 分钟）...")
    r = sh([sys.executable, "-m", "pip", "install", "-q", "vllm", "aiohttp", "torchvision"])
    print("退出码:", r.returncode)
    if r.returncode:
        print(r.stdout[-2500:]); print(r.stderr[-2500:])
else:
    print("三个包都在，跳过安装。")

u = sh([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio"])
print("卸 torchaudio 退出码:", u.returncode)
print()
for p in CHECK + ["transformers", "torch"]:
    print("  %-14s %s" % (p, ver(p) or "（未安装）"))


## 2. 实测可达显存带宽 —— 这是屋顶

不用标称的 320 GB/s。标称值是理论峰值，真实可达值通常是它的 70–85%。
用三个微基准取一个可信区间：

- **copy**：`b.copy_(a)`，读 + 写各一遍
- **read**：`a.sum()`，只读
- **triad**：`c = a + 2.0 * b`，读两遍写一遍（STREAM 风格）

解码时读权重是**纯读**，所以后面用 **read 那个数**当屋顶最贴切。


In [ ]:
import subprocess, sys, io, json

BW = r'''
import torch, time, json
torch.cuda.init()
N = 128 * 1024 * 1024 // 4 * 4        # 512 MB 的 float32
a = torch.ones(N, dtype=torch.float32, device="cuda")
b = torch.ones(N, dtype=torch.float32, device="cuda")
c = torch.empty(N, dtype=torch.float32, device="cuda")
nbytes = a.numel() * 4

def timeit(fn, moved, iters=30):
    for _ in range(5): fn()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(iters): fn()
    torch.cuda.synchronize()
    dt = (time.perf_counter() - t0) / iters
    return moved / dt / 1e9

res = {}
res["copy_GBps"]  = timeit(lambda: b.copy_(a),        nbytes * 2)
res["read_GBps"]  = timeit(lambda: a.sum(),           nbytes)
res["triad_GBps"] = timeit(lambda: torch.add(a, b, alpha=2.0, out=c), nbytes * 3)
res["buffer_MB"] = nbytes / 1e6
print(json.dumps(res))
'''
io.open("bw.py", "w", encoding="utf-8").write(BW)

r = subprocess.run([sys.executable, "bw.py"], capture_output=True, text=True)
line = [l for l in r.stdout.splitlines() if l.startswith("{")]
if line:
    BWRES = json.loads(line[-1])
    print("缓冲区 %.0f MB" % BWRES["buffer_MB"])
    for k in ["copy_GBps", "read_GBps", "triad_GBps"]:
        print("  %-12s %7.1f GB/s   （标称 320 的 %.0f%%）" % (k, BWRES[k], BWRES[k] / 320 * 100))
    print()
    print("后面用 read_GBps = %.1f GB/s 当屋顶（解码读权重是纯读）。" % BWRES["read_GBps"])
else:
    BWRES = None
    print("带宽微基准失败：")
    print((r.stderr or r.stdout)[-2000:])


## 3. 写出压测脚本（与前几轮同一份）

In [ ]:
import io
src = '# -*- coding: utf-8 -*-\n"""vLLM 服务端压测：并发扫描下的吞吐 / TTFT / TPOT，以及前缀复用的效果。\n\n指标定义（与 JD 里那套一致）：\n  TTFT  Time To First Token   —— 首 token 延迟，决定交互体感\n  TPOT  Time Per Output Token —— 首 token 之后的平均出词间隔\n  吞吐   总输出 token 数 / 墙钟时间\n\n用法（先 bash serve.sh 起服务）：\n  python bench_serving.py                 # 并发扫描\n  python bench_serving.py --prefix-test   # 前缀复用对照\n"""\nimport argparse, asyncio, json, statistics as st, time\nimport aiohttp\n\nURL = "http://127.0.0.1:8000/v1/chat/completions"\nMODEL = "Qwen/Qwen2.5-0.5B-Instruct"\n\n# 一段较长的共享 system prompt：开 --enable-prefix-caching 后其 prefill 只算一次\nSHARED_PREFIX = (\n    "You are a meticulous technical assistant. Answer concisely and precisely. "\n    "Always reason step by step before answering. " * 20\n)\n\n\nasync def one_request(sess, prompt, max_tokens, use_prefix):\n    msgs = ([{"role": "system", "content": SHARED_PREFIX}] if use_prefix else []) + \\\n           [{"role": "user", "content": prompt}]\n    body = {"model": MODEL, "messages": msgs, "max_tokens": max_tokens,\n            "temperature": 0.0, "stream": True}\n    t0 = time.perf_counter()\n    ttft, n_tok, last = None, 0, t0\n    async with sess.post(URL, json=body) as resp:\n        async for raw in resp.content:\n            line = raw.decode("utf-8").strip()\n            if not line.startswith("data: ") or line == "data: [DONE]":\n                continue\n            delta = json.loads(line[6:])["choices"][0].get("delta", {})\n            if delta.get("content"):\n                now = time.perf_counter()\n                if ttft is None:\n                    ttft = now - t0\n                n_tok += 1\n                last = now\n    return dict(ttft=ttft or 0.0, total=last - t0, n_tok=n_tok)\n\n\nasync def run_batch(n_conc, n_req, max_tokens, use_prefix):\n    prompts = [f"Explain concept #{i} in distributed systems." for i in range(n_req)]\n    sem = asyncio.Semaphore(n_conc)\n\n    async def guarded(sess, p):\n        async with sem:\n            return await one_request(sess, p, max_tokens, use_prefix)\n\n    timeout = aiohttp.ClientTimeout(total=600)\n    async with aiohttp.ClientSession(timeout=timeout) as sess:\n        await one_request(sess, "warmup", 4, use_prefix)          # 预热\n        t0 = time.perf_counter()\n        rs = await asyncio.gather(*(guarded(sess, p) for p in prompts))\n        wall = time.perf_counter() - t0\n\n    tot_tok = sum(r["n_tok"] for r in rs)\n    tpots = [(r["total"] - r["ttft"]) / max(r["n_tok"] - 1, 1) for r in rs if r["n_tok"] > 1]\n    return dict(conc=n_conc, wall=wall, tput=tot_tok / wall, rps=len(rs) / wall,\n                ttft_p50=st.median(r["ttft"] for r in rs),\n                ttft_p99=sorted(r["ttft"] for r in rs)[int(len(rs) * 0.99) - 1],\n                tpot_p50=st.median(tpots) if tpots else 0.0, tot_tok=tot_tok)\n\n\nasync def sweep(args):\n    print(f"{\'并发\':>5}{\'请求\':>6}{\'墙钟s\':>9}{\'吞吐 tok/s\':>13}{\'RPS\':>8}"\n          f"{\'TTFT p50\':>11}{\'TTFT p99\':>11}{\'TPOT p50\':>11}")\n    print("-" * 74)\n    out = []\n    for c in [1, 2, 4, 8, 16, 32]:\n        r = await run_batch(c, max(c * 4, 16), args.max_tokens, use_prefix=False)\n        print(f"{r[\'conc\']:>5}{max(c*4,16):>6}{r[\'wall\']:>9.2f}{r[\'tput\']:>13.1f}"\n              f"{r[\'rps\']:>8.2f}{r[\'ttft_p50\']*1e3:>10.1f}ms{r[\'ttft_p99\']*1e3:>10.1f}ms"\n              f"{r[\'tpot_p50\']*1e3:>10.2f}ms")\n        out.append(r)\n    json.dump(out, open("sweep_results.json", "w"), indent=1)\n    base = out[0]["tput"]\n    print(f"\\ncontinuous batching 收益：并发 1 → 32，吞吐 "\n          f"{base:.1f} → {out[-1][\'tput\']:.1f} tok/s（{out[-1][\'tput\']/base:.1f}×），"\n          f"TTFT p50 {out[0][\'ttft_p50\']*1e3:.0f} → {out[-1][\'ttft_p50\']*1e3:.0f} ms")\n    print("吞吐与延迟的取舍就在这张表里：并发拉高吞吐涨，但 TTFT 同步恶化。")\n\n\nasync def prefix_test(args):\n    print("前缀复用对照（服务端需带 --enable-prefix-caching 启动）")\n    print(f"{\'场景\':<26}{\'吞吐 tok/s\':>13}{\'TTFT p50\':>12}")\n    print("-" * 51)\n    for label, up in [("无共享前缀", False), (f"共享前缀 ({len(SHARED_PREFIX)} 字符)", True)]:\n        r = await run_batch(8, 32, args.max_tokens, use_prefix=up)\n        print(f"{label:<26}{r[\'tput\']:>13.1f}{r[\'ttft_p50\']*1e3:>11.1f}ms")\n    print("\\n共享前缀命中 KV cache 后，重复的 prefill 不再重算，TTFT 应显著下降。")\n    print("对比未开 --enable-prefix-caching 重启服务再跑一次，差值即为该特性的真实收益。")\n\n\nif __name__ == "__main__":\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--max-tokens", type=int, default=128)\n    ap.add_argument("--prefix-test", action="store_true")\n    a = ap.parse_args()\n    asyncio.run(prefix_test(a) if a.prefix_test else sweep(a))\n'
src = src.replace('MODEL = "Qwen/Qwen2.5-0.5B-Instruct"',
                  'MODEL = open("MODEL_NAME.txt").read().strip()')
io.open("bench_serving.py", "w", encoding="utf-8").write(src)
print("写出 bench_serving.py", len(src), "字符")


## 4. 启动器

抓 `model weights take X GiB` —— 这是 vLLM 报的**实际驻留权重字节**，比 HF 文件大小准。
（三条正则都带捕获组；`except` 只吞连接失败 —— 上一轮在这两点上各踩过一次。）


In [ ]:
import subprocess, sys, time, requests, io, re, json, os

INFO = {}

def serve(model_id, quant, tag, wait=420):
    subprocess.run(["pkill", "-f", "vllm"], check=False)
    time.sleep(12)
    io.open("MODEL_NAME.txt", "w").write(model_id)
    cmd = [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
           "--model", model_id, "--host", "127.0.0.1", "--port", "8000",
           "--max-model-len", "2048", "--gpu-memory-utilization", "0.85",
           "--no-enable-log-requests"]
    if quant:
        cmd += ["--quantization", quant]
    print("启动:", model_id, "| quant =", quant or "fp16")
    lg = open("/content/rf_%s.log" % tag, "w")
    p = subprocess.Popen(cmd, stdout=lg, stderr=subprocess.STDOUT)

    for i in range(wait // 2):
        if p.poll() is not None:
            print("  退出码", p.returncode)
            print(open("/content/rf_%s.log" % tag).read()[-2500:])
            return None
        try:
            if requests.get("http://127.0.0.1:8000/v1/models", timeout=2).status_code == 200:
                print("  就绪，用时 %ds" % (i * 2))
                txt = open("/content/rf_%s.log" % tag).read()
                d = {}
                for pat, key in [(r"model weights take ([\d.]+)\s*GiB", "weights_GiB"),
                                 (r"GPU KV cache size: ([\d,]+)", "kv_tokens"),
                                 (r"(Maximum concurrency[^\n]*)", "concurrency")]:
                    m = re.search(pat, txt)
                    if m:
                        d[key] = m.group(1)
                        print("    %s = %s" % (key, m.group(1)))
                if "weights_GiB" not in d:
                    print("    !! 日志里没抓到 model weights take —— 后面无法算屋顶")
                INFO[tag] = d
                return p
        except requests.RequestException:
            pass
        time.sleep(2)

    print("  %ds 没起来" % wait)
    print(open("/content/rf_%s.log" % tag).read()[-2500:])
    return None


## 5. 三个变体各跑一轮

每轮：起服务 → 抓权重字节 → 并发扫描（拿 TPOT 随并发的曲线）→ 关掉。


In [ ]:
import subprocess, sys, json, os

MODELS = {
    "fp16":      "Qwen/Qwen2.5-1.5B-Instruct",
    "AWQ":       "Qwen/Qwen2.5-1.5B-Instruct-AWQ",
    "GPTQ-Int4": "Qwen/Qwen2.5-1.5B-Instruct-GPTQ-Int4",
}
QUANT = {"fp16": None, "AWQ": "awq", "GPTQ-Int4": "gptq"}

SWEEP = {}
for tag in ["fp16", "AWQ", "GPTQ-Int4"]:
    print("=" * 60)
    p = serve(MODELS[tag], QUANT[tag], tag)
    if not p:
        print("  起服务失败，跳过", tag)
        continue
    r = subprocess.run([sys.executable, "-u", "bench_serving.py"],
                       capture_output=True, text=True)
    print(r.stdout)
    if os.path.exists("sweep_results.json"):
        SWEEP[tag] = json.load(open("sweep_results.json"))
        os.rename("sweep_results.json", "rf_sweep_%s.json" % tag)
print("=" * 60)
print("三个变体完成:", list(SWEEP.keys()))


## 6. 算屋顶，下判断

`TPOT 理论下界 = 权重字节 / read 带宽`，然后看实测 TPOT 是它的几倍。


In [ ]:
import json

if BWRES is None:
    print("没有带宽数据，无法算屋顶。")
else:
    BW = BWRES["read_GBps"] * 1e9      # bytes/s
    print("屋顶用的 read 带宽: %.1f GB/s" % BWRES["read_GBps"])
    print()
    print("%-11s %10s %12s %12s %10s" % ("变体", "权重GiB", "理论下界ms", "实测TPOT ms", "实测/下界"))
    print("-" * 60)
    ratios = {}
    for tag in ["fp16", "AWQ", "GPTQ-Int4"]:
        d = INFO.get(tag, {})
        w = d.get("weights_GiB")
        sw = SWEEP.get(tag)
        tpot = None
        if sw:
            for row in sw:
                if row["conc"] == 1:
                    tpot = row["tpot_p50"] * 1e3
        if w is None or tpot is None:
            print("%-11s %10s %12s %12s %10s" % (tag, w or "-", "-", "%.2f" % tpot if tpot else "-", "-"))
            continue
        wb = float(w) * (1024 ** 3)
        lb = wb / BW * 1e3            # ms
        ratios[tag] = tpot / lb
        print("%-11s %10s %12.2f %12.2f %10.2f" % (tag, w, lb, tpot, ratios[tag]))

    print()
    if len(ratios) >= 2:
        vals = list(ratios.values())
        spread = max(vals) / min(vals)
        print("实测/下界 的离散度（最大÷最小）= %.2f" % spread)
        print()
        if spread <= 1.6:
            print("判据 → 三者都贴着同一条带宽屋顶，比值接近。**假说成立**：")
            print("       解码受权重读取带宽限制，量化靠减少访存取胜。")
        elif ratios.get("fp16", 9) < min([v for k, v in ratios.items() if k != "fp16"] or [9]) / 1.6:
            print("判据 → fp16 贴屋顶而量化版明显偏离。**假说不完整**：")
            print("       反量化开销吃掉了一部分访存收益，不能只用带宽解释。")
        else:
            print("判据 → 比值离散或普遍远离屋顶。**假说存疑**，需要更细的剖析。")
        print()
        print("注意：比值普遍在 1.5–2.5 属正常——屋顶只算了权重，")
        print("      没算 KV cache 读写、激活、kernel 启动与调度开销。")


## 7. 第二个推论：优势随并发收窄的拐点

In [ ]:
print("%-6s %10s %10s %12s %12s" % ("并发", "fp16 TPOT", "AWQ TPOT", "AWQ吞吐/fp16", "GPTQ吞吐/fp16"))
print("-" * 56)
def pick(tag, c, key):
    for row in SWEEP.get(tag, []) or []:
        if row["conc"] == c:
            return row[key]
    return None

for c in [1, 2, 4, 8, 16, 32]:
    f_t, a_t = pick("fp16", c, "tpot_p50"), pick("AWQ", c, "tpot_p50")
    f_q, a_q, g_q = pick("fp16", c, "tput"), pick("AWQ", c, "tput"), pick("GPTQ-Int4", c, "tput")
    print("%-6d %10s %10s %12s %12s" % (
        c,
        "%.2f" % (f_t * 1e3) if f_t else "-",
        "%.2f" % (a_t * 1e3) if a_t else "-",
        "%.2f×" % (a_q / f_q) if (a_q and f_q) else "-",
        "%.2f×" % (g_q / f_q) if (g_q and f_q) else "-"))

print()
print("读法：若低并发受带宽限制、高并发转为受算力限制，")
print("      量化的吞吐优势应当随并发单调收窄，TPOT 曲线应当在某个并发之后明显抬头。")
print("      拐点出现在哪一档，就是这块卡上「带宽受限 → 算力受限」的分界。")


## 8. 边界

- 屋顶只计权重读取，**没有**计入 KV cache 读写、激活张量、kernel 启动与调度开销。
  所以「实测/下界」天然会大于 1，关注的是**三个变体之间这个比值是否接近**，不是它的绝对值。
- 带宽微基准用的是大块连续访问，是**乐观上界**；权重读取有布局与 tile 的额外开销。
- 单次测量。若某个变体的比值明显异常，先重复跑再下结论。
- 这不能替代 nsight/ncu 的真实剖析，只是一个**可自证的量级检验**。
  面试若被追问，要说清楚这是 roofline 量级估算，不是 kernel 级剖析。
